In [2]:
!pip install fastapi uvicorn pyngrok nest-asyncio transformers bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.1 MB/s eta 0:00:00:00:0100:01


In [3]:
!pip install pypdf langchain Tika

In [4]:
!pip install -U langchain-text-splitters langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.1/168.1 kB 11.5 MB/s eta 0:00:00
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.0.9
    Uninstalling langgraph-1.0.9:
      Successfully uninstalled langgraph-1.0.9
  Attempting uninstall: langchain
    Found existing installation: langchain 1.2.10
    Uninstalling langchain-1.2.10:
      Successfully uninstalled langchain-1.2.10


In [1]:
from google.colab import userdata
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
from pyngrok import ngrok

auth_token = user_secrets.get_secret("NGROK_TOKEN")
ngrok.set_auth_token(auth_token)

# Clean up any existing tunnels
tunnels = ngrok.get_tunnels()
for tunnel in tunnels:
    ngrok.disconnect(tunnel.public_url)

public_url = ngrok.connect(8000).public_url
print(f"✅ Your Backend API will be live at: {public_url}")

✅ Your Backend API will be live at: https://unconvicting-jeffry-doable.ngrok-free.dev


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import os
import gc

# 1. HARD RESET for GPU Memory
# This clears any data left behind by the crashed kernel
def clear_memory():
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.ipc_collect()

clear_memory()

# 2. Use Kaggle's local working directory
KAGGLE_MODEL_PATH = "/kaggle/working/Mistral-V2"
os.makedirs(KAGGLE_MODEL_PATH, exist_ok=True)

model_id = "filipealmeida/Mistral-7B-Instruct-v0.1-sharded"

# 3. Optimized 4-bit Config for Kaggle T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16 # Changed from bfloat16 for Kaggle stability
)

print(f"⏳ Loading {model_id} into Kaggle GPU...")

# 4. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_id, 
    trust_remote_code=True, 
    cache_dir=KAGGLE_MODEL_PATH
)
tokenizer.pad_token = tokenizer.eos_token

# 5. Load Model with Memory Guards
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    quantization_config=bnb_config, 
    device_map="auto",
    trust_remote_code=True,
    cache_dir=KAGGLE_MODEL_PATH,
    torch_dtype=torch.float16,   # Matches T4 architecture better
    low_cpu_mem_usage=True,      # Reduces RAM spikes
    offload_folder="offload"     # Fallback if VRAM gets tight
)

print("✅ Model Ready! GPU RAM usage should be around 5.5GB.")

⏳ Loading filipealmeida/Mistral-7B-Instruct-v0.1-sharded into Kaggle GPU...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

✅ Model Ready! GPU RAM usage should be around 5.5GB.


In [3]:
from fastapi import FastAPI, UploadFile, File
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader
import io, nest_asyncio, uvicorn, asyncio, json, re

app = FastAPI()

# ─────────────────────────────────────────────
# Make sure model & tokenizer are loaded before
# the server starts (do this ONCE at module level)
# ─────────────────────────────────────────────
# from transformers import AutoTokenizer, AutoModelForCausalLM
# import torch
# MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"   # change to your model
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# model     = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16).to("cuda")


# ── helpers ──────────────────────────────────

def process_pdf(file_bytes: bytes) -> list[str]:
    """Extract text from PDF and return chunks."""
    try:
        reader = PdfReader(io.BytesIO(file_bytes))
        full_text = ""
        for page in reader.pages:
            content = page.extract_text()
            if content:
                full_text += content + "\n"

        if not full_text.strip():
            return []

        # Larger chunks = more context per question set
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=2000,
            chunk_overlap=200,
        )
        return splitter.split_text(full_text)

    except Exception as e:
        print(f"[PDF Error] {e}")
        return []


def build_context(chunks: list[str], max_chars: int = 3000) -> str:
    """
    Merge several chunks up to `max_chars` so the model has
    enough document content to write grounded questions.
    """
    context = ""
    for chunk in chunks:
        if len(context) + len(chunk) > max_chars:
            break
        context += chunk + "\n\n"
    return context.strip()


def build_prompt(context: str, num_questions: int = 5) -> str:
    """
    Instruction-tuned prompt that forces the model to stay
    grounded in the supplied document text.
    """
    return f"""[INST] You are a quiz generator. Your ONLY job is to create multiple-choice questions
STRICTLY based on the document excerpt provided below.

Rules:
- Every question MUST come directly from facts stated in the document.
- Do NOT use outside knowledge or invent facts.
- Each question must have exactly 4 options labeled A, B, C, D.
- The answer field must be the full text of the correct option.
- Return ONLY a valid JSON array — no extra text, no markdown fences.

JSON format (return exactly this structure):
[
  {{
    "question": "...",
    "options": ["A. ...", "B. ...", "C. ...", "D. ..."],
    "answer": "A. ..."
  }}
]

Document:
\"\"\"
{context}
\"\"\"

Generate {num_questions} multiple-choice questions from the document above. [/INST]"""


def clean_and_parse_json(raw: str) -> list | None:
    """Strip noise and parse the JSON list the model returned."""
    # Remove markdown code fences if present
    raw = re.sub(r"```(?:json)?", "", raw).strip()

    # Isolate the first JSON array found
    start = raw.find("[")
    end   = raw.rfind("]") + 1
    if start == -1 or end == 0:
        return None

    try:
        return json.loads(raw[start:end])
    except json.JSONDecodeError:
        # Last resort: try to fix truncated JSON by closing open structures
        snippet = raw[start:end]
        try:
            # Sometimes generation cuts off mid-object; try a partial parse
            fixed = snippet.rstrip(",").rstrip() + "]"
            return json.loads(fixed)
        except Exception:
            return None


# ── route ────────────────────────────────────

@app.post("/generate-quiz")
async def generate_quiz(
    file: UploadFile = File(...),
    num_questions: int = 5,
):
    try:
        file_bytes = await file.read()
        chunks = process_pdf(file_bytes)

        if not chunks:
            return {"error": "Could not extract text from PDF. Check that the file is not scanned/image-only."}

        # ── FIX 1: use multiple chunks, not just chunks[0] ──
        context = build_context(chunks, max_chars=3000)

        print(f"\n[Context sent to model — {len(context)} chars]\n{context[:500]}...\n")

        # ── FIX 2: grounded, explicit prompt ──
        prompt = build_prompt(context, num_questions=num_questions)

        # ── generation ──
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        input_length = inputs.input_ids.shape[1]

        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,          # enough for 5 detailed MCQs
            min_new_tokens=100,
            do_sample=True,
            temperature=0.3,              # LOW temp = more faithful to document
            top_p=0.9,
            top_k=40,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id,
        )

        response_tokens = outputs[0][input_length:]
        raw_output = tokenizer.decode(response_tokens, skip_special_tokens=True).strip()

        print("\n" + "=" * 60)
        print(f"RAW MODEL OUTPUT:\n{raw_output}")
        print("=" * 60 + "\n")

        if not raw_output:
            return {"error": "Model generated no output. Try reducing num_questions or PDF size."}

        # ── FIX 3: robust JSON parsing ──
        quiz_data = clean_and_parse_json(raw_output)

        if quiz_data is not None:
            return {"quiz": quiz_data, "chunks_used": len(chunks), "context_chars": len(context)}

        # Fallback: return raw text so the client can still display something
        return {
            "quiz": raw_output,
            "format": "raw_text",
            "note": "JSON parsing failed — returning raw model output.",
        }

    except Exception as e:
        print(f"[Server Error] {e}")
        return {"error": str(e)}


# ── entry point ──────────────────────────────

if __name__ == "__main__":
    nest_asyncio.apply()
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
    server = uvicorn.Server(config)
    loop = asyncio.get_event_loop()
    loop.create_task(server.serve())

INFO:     Started server process [716]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)



[Context sent to model — 1966 chars]
Pollution refers to the introduction of harmful substances or contaminants into the natural 
environment, causing adverse changes to ecosystems, human health, and the planet's 
balance. It manifests in various forms, including **air**, **water**, **soil**, **noise**, 
**light**, and **plastic pollution**. Human activities—such as industrialization, 
urbanization, agriculture, transportation, and poor waste management—are the primary 
drivers. While pollution has existed since early human civiliz...


RAW MODEL OUTPUT:
Here are five multiple-choice questions generated based on the provided document:

1. What is pollution?
a) The introduction of beneficial substances or contaminants into the natural environment
b) The presence of harmful substances or contaminants in the natural environment
c) The absence of any substances or contaminants in the natural environment
d) None of the above
Answer: b) The presence of harmful substances or contaminants in 